In [1]:
!pip install -q transformers scikit-learn pandas==2.2.2 librosa soundfile matplotlib

## Imports

In [2]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import librosa
import torch
import soundfile as sf

from google.colab import files
from transformers import (
    AutoFeatureExtractor,
    WavLMModel,
    Wav2Vec2Model,
    HubertModel,
)
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")

## Config

In [3]:
TARGET_SR = 16000
CHUNK_SEC = 1.5
HOP_SEC = 0.75
MIN_SEGMENT_SEC = 0.6

BASE_DIR = "/content/speaker_diarization_project"
RAW_DIR = os.path.join(BASE_DIR, "raw_audio")
OUT_DIR = os.path.join(BASE_DIR, "outputs")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Detected device:", device)

Detected device: cuda


## Optional human verification mapping

Edit this if your files or human counts change.

Current mapping matches your latest labels:
1 → 1  
2 → 1  
3 → 0  
4 → 0  
5 → 0  
6 → 2  
7 → 0  
8 → 3  
9 → 3

In [4]:
FILE_ORDER = [
    "1 (1).wav",
    "1.wav",
    "10convert.com_Audience-Claps_daSG5fwdA7o.wav",
    "doing_the_dishes.wav",
    "dude_miaowing.wav",
    "meeting-clip2.wav",
    "pink_noise.wav",
    "test_1.wav",
    "test_2.wav"
]

HUMAN_COUNT_MAP_BY_SERIAL = {
    1: 1,
    2: 1,
    3: 0,
    4: 0,
    5: 0,
    6: 2,
    7: 0,
    8: 3,
    9: 3
}

FILE_HUMAN_MAP = {
    file_name: HUMAN_COUNT_MAP_BY_SERIAL[i + 1]
    for i, file_name in enumerate(FILE_ORDER)
}

# obvious non-speech files
NON_SPEECH_FILES = {
    "10convert.com_Audience-Claps_daSG5fwdA7o.wav",
    "doing_the_dishes.wav",
    "dude_miaowing.wav",
    "pink_noise.wav"
}

## Upload audio files from your Mac

In [5]:
print("Upload your audio files...")
uploaded = files.upload()

audio_paths = []
for fname, data in uploaded.items():
    save_path = os.path.join(RAW_DIR, fname)
    with open(save_path, "wb") as f:
        f.write(data)
    audio_paths.append(save_path)

audio_paths = sorted(audio_paths)

print("Uploaded files:")
for p in audio_paths:
    print("-", os.path.basename(p))

Upload your audio files...


Saving 1 (1).wav to 1 (1).wav
Saving 1.wav to 1.wav
Saving 10convert.com_Audience-Claps_daSG5fwdA7o.wav to 10convert.com_Audience-Claps_daSG5fwdA7o.wav
Saving doing_the_dishes.wav to doing_the_dishes.wav
Saving dude_miaowing.wav to dude_miaowing.wav
Saving meeting-clip2.wav to meeting-clip2.wav
Saving pink_noise.wav to pink_noise.wav
Saving test_1.wav to test_1.wav
Saving test_2.wav to test_2.wav
Uploaded files:
- 1 (1).wav
- 1.wav
- 10convert.com_Audience-Claps_daSG5fwdA7o.wav
- doing_the_dishes.wav
- dude_miaowing.wav
- meeting-clip2.wav
- pink_noise.wav
- test_1.wav
- test_2.wav


## Load models

In [6]:
print("Loading WavLM...")
wavlm_extractor = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base")
wavlm_model = WavLMModel.from_pretrained("microsoft/wavlm-base")
wavlm_model.eval()

print("Loading Wav2Vec2...")
wav2vec2_extractor = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base-960h")
wav2vec2_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")
wav2vec2_model.eval()

print("Loading HuBERT...")
hubert_extractor = AutoFeatureExtractor.from_pretrained("facebook/hubert-base-ls960")
hubert_model = HubertModel.from_pretrained("facebook/hubert-base-ls960")
hubert_model.eval()

print("All models loaded successfully.")

Loading WavLM...


preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

...

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

All models loaded successfully.


## Helpers

In [7]:
def load_audio_mono(path, sr=TARGET_SR):
    audio, sample_rate = librosa.load(path, sr=sr, mono=True)
    return audio, sample_rate

def normalize_audio(audio):
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak
    return audio

def split_audio(audio, sr, chunk_sec=CHUNK_SEC, hop_sec=HOP_SEC):
    chunk_size = int(chunk_sec * sr)
    hop_size = int(hop_sec * sr)
    chunks = []
    seg_id = 0

    for start in range(0, len(audio), hop_size):
        end = min(start + chunk_size, len(audio))
        if end - start < int(0.3 * sr):
            continue

        chunks.append({
            "segment_id": seg_id,
            "start_time": start / sr,
            "end_time": end / sr,
            "audio": audio[start:end]
        })
        seg_id += 1

    return chunks

def is_speech_chunk(chunk_audio, sr=TARGET_SR):
    if len(chunk_audio) == 0:
        return False

    rms = np.sqrt(np.mean(np.square(chunk_audio)))
    if rms < 0.008:
        return False

    zcr = np.mean(librosa.feature.zero_crossing_rate(y=chunk_audio))
    if zcr > 0.20:
        return False

    spectral_centroid = np.mean(librosa.feature.spectral_centroid(y=chunk_audio, sr=sr))
    if spectral_centroid > 2800:
        return False

    rolloff = np.mean(librosa.feature.spectral_rolloff(y=chunk_audio, sr=sr))
    if rolloff > 3800:
        return False

    return True

## Embedding extraction

In [8]:
def get_wavlm_embedding(chunk_audio, sr=TARGET_SR, run_device="cpu"):
    model = wavlm_model.to(run_device)

    inputs = wavlm_extractor(
        chunk_audio,
        sampling_rate=sr,
        return_tensors="pt",
        padding=True
    )
    inputs = {k: v.to(run_device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        emb = outputs.last_hidden_state.mean(dim=1).squeeze().detach().cpu().numpy()

    return emb

def get_wav2vec2_embedding(chunk_audio, sr=TARGET_SR, run_device="cpu"):
    model = wav2vec2_model.to(run_device)

    inputs = wav2vec2_extractor(
        chunk_audio,
        sampling_rate=sr,
        return_tensors="pt",
        padding=True
    )
    inputs = {k: v.to(run_device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        emb = outputs.last_hidden_state.mean(dim=1).squeeze().detach().cpu().numpy()

    return emb

def get_hubert_embedding(chunk_audio, sr=TARGET_SR, run_device="cpu"):
    model = hubert_model.to(run_device)

    inputs = hubert_extractor(
        chunk_audio,
        sampling_rate=sr,
        return_tensors="pt",
        padding=True
    )
    inputs = {k: v.to(run_device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        emb = outputs.last_hidden_state.mean(dim=1).squeeze().detach().cpu().numpy()

    return emb

## Clustering

In [9]:
def cluster_embeddings(embeddings, max_speakers=4):
    if len(embeddings) == 0:
        return np.array([])

    if len(embeddings) == 1:
        return np.array([0])

    X = np.array(embeddings)
    X = StandardScaler().fit_transform(X)

    best_labels = None
    best_score = -1

    max_k = min(max_speakers, len(X))

    for k in range(2, max_k + 1):
        try:
            clustering = AgglomerativeClustering(n_clusters=k)
            labels = clustering.fit_predict(X)

            if len(set(labels)) < 2:
                continue

            score = silhouette_score(X, labels)

            # prefer larger k only if it is clearly better
            if score > best_score + 0.02:
                best_score = score
                best_labels = labels

        except Exception:
            continue

    if best_labels is None:
        return np.zeros(len(X), dtype=int)

    return best_labels

## Main diarization function

In [10]:
def empty_result_df():
    return pd.DataFrame(columns=[
        "file_name",
        "start_time",
        "end_time",
        "speaker_number",
        "model_name",
        "device",
        "human_verification_speaker"
    ])

def diarize_with_transformer_model(audio_path, model_name="wavlm", run_device="cpu"):
    start_runtime = time.time()
    file_name = os.path.basename(audio_path)

    # hard block clear non-speech files
    if file_name in NON_SPEECH_FILES:
        runtime = time.time() - start_runtime
        return empty_result_df(), runtime

    audio, sr = load_audio_mono(audio_path, TARGET_SR)
    audio = normalize_audio(audio)
    chunks = split_audio(audio, sr)

    embeddings = []
    valid_meta = []

    for c in chunks:
        seg_audio = c["audio"]
        duration = c["end_time"] - c["start_time"]

        if not is_speech_chunk(seg_audio, sr):
            continue

        if duration < MIN_SEGMENT_SEC:
            continue

        if model_name == "wavlm":
            emb = get_wavlm_embedding(seg_audio, sr=sr, run_device=run_device)
        elif model_name == "wav2vec2":
            emb = get_wav2vec2_embedding(seg_audio, sr=sr, run_device=run_device)
        elif model_name == "hubert":
            emb = get_hubert_embedding(seg_audio, sr=sr, run_device=run_device)
        else:
            raise ValueError(f"Unsupported model: {model_name}")

        embeddings.append(emb)
        valid_meta.append(c)

    if len(embeddings) == 0:
        runtime = time.time() - start_runtime
        return empty_result_df(), runtime

    labels = cluster_embeddings(embeddings, max_speakers=4)

    rows = []
    for i, c in enumerate(valid_meta):
        rows.append({
            "file_name": file_name,
            "start_time": round(c["start_time"], 2),
            "end_time": round(c["end_time"], 2),
            "speaker_number": f"speaker_{int(labels[i]) + 1}",
            "model_name": model_name,
            "device": run_device,
            "human_verification_speaker": ""
        })

    runtime = time.time() - start_runtime
    df = pd.DataFrame(rows)
    return df, runtime

## Run comparison

In [11]:
required_models = ["wavlm_model", "wav2vec2_model", "hubert_model"]
for m in required_models:
    if m not in globals():
        raise ValueError(f"{m} is not loaded. Run the model loading cell first.")

all_results = []
summary_rows = []

devices_to_test = ["cpu"]
if torch.cuda.is_available():
    devices_to_test.append("cuda")

models_to_test = ["wavlm", "wav2vec2", "hubert"]

for audio_path in audio_paths:
    print(f"\nProcessing: {os.path.basename(audio_path)}")

    for run_device in devices_to_test:
        print(f"  Device: {run_device}")

        for model_name in models_to_test:
            try:
                df_model, runtime = diarize_with_transformer_model(
                    audio_path,
                    model_name=model_name,
                    run_device=run_device
                )

                all_results.append(df_model)

                summary_rows.append({
                    "file_name": os.path.basename(audio_path),
                    "model_name": model_name,
                    "device": run_device,
                    "segments_found": len(df_model),
                    "speakers_found": df_model["speaker_number"].nunique() if not df_model.empty else 0,
                    "runtime_sec": round(runtime, 2)
                })

                print(f"    {model_name} done in {runtime:.2f}s")

            except Exception as e:
                print(f"    {model_name} failed: {e}")


Processing: 1 (1).wav
  Device: cpu
    wavlm done in 14.25s
    wav2vec2 done in 0.00s
    hubert done in 0.00s
  Device: cuda
    wavlm done in 0.00s
    wav2vec2 done in 0.00s
    hubert done in 0.00s

Processing: 1.wav
  Device: cpu
    wavlm done in 0.64s
    wav2vec2 done in 0.25s
    hubert done in 0.25s
  Device: cuda
    wavlm done in 1.90s
    wav2vec2 done in 0.14s
    hubert done in 0.15s

Processing: 10convert.com_Audience-Claps_daSG5fwdA7o.wav
  Device: cpu
    wavlm done in 0.00s
    wav2vec2 done in 0.00s
    hubert done in 0.00s
  Device: cuda
    wavlm done in 0.00s
    wav2vec2 done in 0.00s
    hubert done in 0.00s

Processing: doing_the_dishes.wav
  Device: cpu
    wavlm done in 0.00s
    wav2vec2 done in 0.00s
    hubert done in 0.00s
  Device: cuda
    wavlm done in 0.00s
    wav2vec2 done in 0.00s
    hubert done in 0.00s

Processing: dude_miaowing.wav
  Device: cpu
    wavlm done in 0.00s
    wav2vec2 done in 0.00s
    hubert done in 0.00s
  Device: cuda
    w

## Create the detailed diarization CSV

In [12]:
if len(all_results) > 0:
    results_df = pd.concat(all_results, ignore_index=True)
else:
    results_df = empty_result_df()

# fill detailed CSV human verification using WavLM per-segment where possible
if not results_df.empty:
    results_df["start_time"] = results_df["start_time"].round(2)
    results_df["end_time"] = results_df["end_time"].round(2)

    wavlm_df = results_df[results_df["model_name"] == "wavlm"].copy()
    merged_df = results_df.merge(
        wavlm_df[["file_name", "start_time", "end_time", "speaker_number"]],
        on=["file_name", "start_time", "end_time"],
        how="left",
        suffixes=("", "_wavlm")
    )
    merged_df["human_verification_speaker"] = merged_df["speaker_number_wavlm"]
    merged_df = merged_df.drop(columns=["speaker_number_wavlm"])
    results_df = merged_df

results_csv = os.path.join(OUT_DIR, "speaker_diarization_results.csv")
results_df.to_csv(results_csv, index=False)

print("Saved detailed CSV:", results_csv)
results_df.head(20)

Saved detailed CSV: /content/speaker_diarization_project/outputs/speaker_diarization_results.csv


,file_name,start_time,end_time,speaker_number,model_name,device,human_verification_speaker
0,1.wav,0.00,1.00,speaker_1,wavlm,cpu,speaker_1
1,1.wav,0.00,1.00,speaker_1,wavlm,cpu,speaker_1
2,1.wav,0.00,1.00,speaker_1,wav2vec2,cpu,speaker_1
3,1.wav,0.00,1.00,speaker_1,wav2vec2,cpu,speaker_1
4,1.wav,0.00,1.00,speaker_1,hubert,cpu,speaker_1
5,1.wav,0.00,1.00,speaker_1,hubert,cpu,speaker_1
6,1.wav,0.00,1.00,speaker_1,wavlm,cuda,speaker_1
7,1.wav,0.00,1.00,speaker_1,wavlm,cuda,speaker_1
8,1.wav,0.00,1.00,speaker_1,wav2vec2,cuda,speaker_1
9,1.wav,0.00,1.00,speaker_1,wav2vec2,cuda,speaker_1


## Create the final summary CSV

In [13]:
records = []

for audio_path in audio_paths:
    file_name = os.path.basename(audio_path)

    audio, sr = load_audio_mono(audio_path)
    full_duration = round(len(audio) / sr, 2)

    for run_device in devices_to_test:
        for model_name in ["wavlm", "wav2vec2", "hubert"]:

            row = next(
                (r for r in summary_rows
                 if r["file_name"] == file_name
                 and r["model_name"] == model_name
                 and r["device"] == run_device),
                None
            )

            if row is None:
                continue

            records.append({
                "file_name": file_name,
                "duration": full_duration,
                "model_name": model_name,
                "device_used": run_device,
                "number_of_speakers": row["speakers_found"],
                "processing_time_sec": row["runtime_sec"],
                "human_verification_speaker": FILE_HUMAN_MAP.get(file_name)
            })

final_analysis_df = pd.DataFrame(records)

analysis_csv = os.path.join(OUT_DIR, "model_speaker_analysis_corrected.csv")
final_analysis_df.to_csv(analysis_csv, index=False)

print("Saved summary CSV:", analysis_csv)
final_analysis_df.head(20)

Saved summary CSV: /content/speaker_diarization_project/outputs/model_speaker_analysis_corrected.csv


,file_name,duration,model_name,device_used,number_of_speakers,processing_time_sec,human_verification_speaker
0,1 (1).wav,1.00,wavlm,cpu,0,14.25,1
1,1 (1).wav,1.00,wav2vec2,cpu,0,0.00,1
2,1 (1).wav,1.00,hubert,cpu,0,0.00,1
3,1 (1).wav,1.00,wavlm,cuda,0,0.00,1
4,1 (1).wav,1.00,wav2vec2,cuda,0,0.00,1
5,1 (1).wav,1.00,hubert,cuda,0,0.00,1
6,1.wav,1.00,wavlm,cpu,1,0.64,1
7,1.wav,1.00,wav2vec2,cpu,1,0.25,1
8,1.wav,1.00,hubert,cpu,1,0.25,1
9,1.wav,1.00,wavlm,cuda,1,1.90,1


## Download the 2 useful CSV files

In [14]:
files.download(results_csv)
files.download(analysis_csv)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>